# Triển khai trên Google Colab – Version 0 (CPU tuần tự)

V0 chỉ chạy trên CPU nên build bằng `g++`, không cần GPU. Chọn `Runtime → Run all`.

- Mã nguồn: clone từ repository [`Panorama-Image-Stitching`](https://github.com/ZelusIsMe-05/Panorama-Image-Stitching) trên GitHub (repo public).
- Dataset: tải thẳng từ thư mục Google Drive bằng `gdown` (thư mục phải chia sẻ *Anyone with the link*), lưu vào `/content/dataset`.
- `PAIR_DIR`: đường dẫn tới **thư mục** của một cặp ảnh. Chương trình tự lấy 2 file `*.png` trong thư mục đó, sắp theo tên: file đứng trước là ảnh trái (A), file đứng sau là ảnh phải (B).

In [ ]:
REPO_URL = "https://github.com/ZelusIsMe-05/Panorama-Image-Stitching.git"
DATASET_URL = "https://drive.google.com/drive/folders/1eWi9OBGeIYzuW7wOhOMybv0-ZHNS6YrO"
RUNS = 5                                                              # số lần đo (+1 lần warm-up)

## 1. Lấy mã nguồn và dữ liệu

In [ ]:
import glob, os, re, subprocess
from IPython.display import Image, display
%cd -q /content

print("Đang clone mã nguồn từ GitHub...")
!rm -rf project && git clone -q --depth 1 {REPO_URL} project
print("  -> Xong: /content/project")

if os.path.isdir("dataset"):
    print("Dataset đã có trong /content/dataset, bỏ qua bước tải.")
else:
    print("Đang tải dataset từ Google Drive...")
    !gdown -q --folder {DATASET_URL} -O dataset --remaining-ok

PAIR_DIRS = sorted(d for d in {os.path.dirname(p) for p in glob.glob("/content/dataset/**/*.png", recursive=True)}
                   if len(glob.glob(os.path.join(d, "*.png"))) == 2)
print(f"  -> Xong: {len(PAIR_DIRS)} cặp ảnh trong /content/dataset")

## 2. Build chương trình

In [ ]:
!g++ -O3 -std=c++17 -x c++ project/src/main.cu -x none project/src/*.cpp -o stitch && echo "BUILD OK: /content/stitch"

## 3. Ghép cặp chỉ định

`pair_images` lấy 2 file `*.png` trong `PAIR_DIR` (báo lỗi nếu thư mục không có đúng 2 ảnh), rồi gọi
`./stitch <trái> <phải> <output> --version 0 --runs RUNS`. In bảng thời gian từng stage và ảnh sau khi ghép.

In [ ]:
PAIR_DIR = "/content/dataset/medium/standard pairs/Outdoor/Pair 1"   # thư mục chứa 2 ảnh .png cần ghép

def pair_images(pair_dir):
    imgs = sorted(glob.glob(os.path.join(pair_dir, "*.png")))
    assert len(imgs) == 2, f"'{pair_dir}' cần có đúng 2 ảnh .png (đang có {len(imgs)})"
    return imgs

def stitch(pair_dir, out):
    left, right = pair_images(pair_dir)
    return subprocess.run(["./stitch", left, right, out, "--version", "0", "--runs", str(RUNS)],
                          capture_output=True, text=True)

left, right = pair_images(PAIR_DIR)
print("Ảnh trái :", os.path.basename(left))
display(Image(filename=left, width=700))
print("Ảnh phải :", os.path.basename(right))
display(Image(filename=right, width=700))
p = stitch(PAIR_DIR, "/content/panorama.jpg")
if p.returncode == 0:
    lines = p.stdout.splitlines()
    print("\n".join(lines[next(i for i, l in enumerate(lines) if l.startswith("Image size")):]))
    print("\nẢnh panorama:")
    display(Image("/content/panorama.jpg", width=900))
else:
    print(p.stdout, p.stderr)

## 4. Chạy toàn bộ dataset

Tổng thời gian pipeline (trung bình mỗi cặp, không tính đọc/ghi ảnh) của tất cả các cặp.

In [ ]:
total_ms, failed = 0.0, []
for d in PAIR_DIRS:
    p = stitch(d, "/content/tmp.jpg")
    m = re.search(r"Total \(end-to-end\)\s+([\d.]+)", p.stdout)
    if p.returncode == 0 and m:
        total_ms += float(m.group(1))
    else:
        failed.append(d)
print(f"{len(PAIR_DIRS) - len(failed)}/{len(PAIR_DIRS)} cặp thành công, tổng thời gian: {total_ms:.1f} ms ({total_ms / 1000:.2f} s)")
if failed:
    print("Lỗi:", *failed, sep="\n  ")